In [1]:
import json
import os
import sys

import numpy as np
import torch

from rctorch.models import MorrisLecarCurrent
from rctorch.optimizers import BruteForceMesh, KWArgsEncoder
from rctorch.supervisors import LorenzAttractor
from rctorch.utils import minmax_transform

In [2]:
# Define the output directory
output_dir = os.path.join(os.getcwd(), "bfm_output", "test_bfm")
# Define the directory to save results
os.makedirs(output_dir, exist_ok=True)

seed = 1
np.random.seed(seed)
torch.cuda.manual_seed(seed)
torch.random.manual_seed(seed)

In [3]:
# Generate the supervisor signal
T = 5_000
dt = 1e-1
x = LorenzAttractor(T, dt, tau=0.01).generate(transient_time=500.0)

x = x.T
sup = minmax_transform(x, zero_mean=True)
print(sup.shape)
np.save(os.path.join(output_dir, "supervisor.npy"), sup)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sup_tensor = torch.tensor(sup, device=device, dtype=torch.float32)

(50000, 3)


In [4]:
# Set the model parameters
Ne = 400
Ni = 100
N = Ne + Ni
BIAS = np.ones((N, 1)) * 65.0
reservoir_params = {
    "model_cls": MorrisLecarCurrent,
    "n_input": sup_tensor.size(1),
    "n_output": sup_tensor.size(1),
    "Ne": Ne,
    "Ni": Ni,
    "dt": dt,
    "BIAS": BIAS,
    "p_sparsity": 0.1,
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}


params_file = os.path.join(output_dir, "reservoir_params.json")
with open(params_file, "w") as f:
    json.dump(
        reservoir_params, f, cls=KWArgsEncoder, indent=4
    )  # Added indent for readability

q_range = np.array([20, 400])
gbar_range = np.array([5, 100])

opt_params = {"w_in_amp": q_range, "gbar": gbar_range}
opt_params_file = os.path.join(output_dir, "opt_params.json")
with open(opt_params_file, "w") as f:
    json.dump(
        opt_params, f, cls=KWArgsEncoder, indent=4
    )  # Added indent for readability

size = 1
for _, value_range in opt_params.items():
    size *= value_range.size

print(f"total size of the mesh is: {size}")

train_test_split = 0.5
nt_split = int(sup_tensor.size(0) * train_test_split)
sup_train = sup_tensor[:nt_split]
sup_test = sup_tensor[nt_split:]
train_kwargs = dict(
    x=sup_train,
    nt_transient=int(500 / dt),
    rls_step=20,
    ridge_reg=1.0,
    ff_coeff=1.0,
)
test_kwargs = {
    "x": sup_test,
    "nt_transient": 0,
    "closed_loop": True,
}

n_threads = 4

total size of the mesh is: 4


In [5]:
bfm = BruteForceMesh(
    reservoir_kwargs=reservoir_params,
    train_kwargs=train_kwargs,
    test_kwargs=test_kwargs,
    params=opt_params,
    num_threads=n_threads,
)

bfm.run(output_dir)
print(f"Results saved in: {output_dir}")


Submitted simulation with params: {'w_in_amp': np.int64(20), 'gbar': np.int64(5), 'model_cls': <class 'rctorch.models.morris_lecar.MorrisLecarCurrent'>, 'n_input': 3, 'n_output': 3, 'Ne': 400, 'Ni': 100, 'dt': 0.1, 'BIAS': array([[65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       [65.],
       

  0%|          | 0/5000 [00:00<?, ?it/s]


  1%|          | 41/5000 [00:00<00:12, 400.58it/s]


  2%|▏         | 113/5000 [00:00<00:08, 583.68it/s]


  4%|▎         | 183/5000 [00:00<00:07, 632.37it/s]


  5%|▌         | 254/5000 [00:00<00:07, 659.41it/s]


  6%|▋         | 321/5000 [00:00<00:07, 661.85it/s]




  8%|▊         | 388/5000 [00:00<00:07, 656.45it/s]


  9%|▉         | 456/5000 [00:00<00:06, 658.71it/s]


 11%|█         | 527/5000 [00:00<00:06, 672.68it/s]


 12%|█▏        | 597/5000 [00:00<00:06, 680.59it/s]


 13%|█▎        | 667/5000 [00:01<00:06, 684.48it/s]


 15%|█▍        | 737/5000 [00:01<00:06, 685.50it/s]


 16%|█▌        | 808/5000 [00:01<00:06, 689.83it/s]



 18%|█▊        | 877/5000 [00:01<00:06, 669.15it/s]


 19%|█▉        | 945/5000 [00:01<00:06, 670.91it/s]


 20%|██        | 1013/5000 [00:01<00:05, 673.02it/s]


 22%|██▏       | 1083/5000 [00:01<00:05, 679.53it/s]


 23%|██▎       | 1152/5000 [00:01<00:05, 678.67it/s]


 24%|██▍       | 1221/5000 [00:01<

Transient Period:


0it [00:00, ?it/s]
  0%|          | 0/25000 [00:00<?, ?it/s]

100%|██████████| 25000/25000 [00:41<00:00, 608.93it/s]


Transient Period:


...

Transient Period:




0it [00:00, ?it/s]


100%|██████████| 25000/25000 [00:41<00:00, 607.91it/s]


Transient Period:


0it [00:00, ?it/s]



  0%|          | 114/25000 [00:00<00:43, 575.17it/s]


  1%|          | 174/25000 [00:00<00:42, 582.83it/s]


  1%|          | 233/25000 [00:00<00:42, 585.13it/s]


  1%|          | 292/25000 [00:00<00:42, 585.03it/s]


  1%|▏         | 351/25000 [00:00<00:42, 584.95it/s]


  2%|▏         | 414/25000 [00:00<00:41, 598.02it/s]


  2%|▏         | 476/25000 [00:00<00:40, 603.96it/s]


  2%|▏         | 537/25000 [00:00<00:40, 600.57it/s]


  2%|▏         | 598/25000 [00:01<00:40, 600.72it/s]


  3%|▎         | 660/25000 [00:01<00:40, 603.83it/s]


  3%|▎         | 721/25000 [00:01<00:40, 603.38it/s]


  3%|▎         | 783/25000 [00:01<00:39, 606.97it/s]


  3%|▎         | 846/25000 [00:01<00:39, 612.01it/s]


  4%|▎         | 908/25000 [00:01<00:39, 606.67it/s]


  4%|▍         | 971/25000 [00:01<00:39, 611.49it/s]


  4%|▍         | 1033/25000 [00:01<00:39, 612.66it/s]


  4%|▍         | 1095/25000 [00:01<00:39, 605.99it/s]


  5%|▍         | 1156/25000 [00:01<00:39,

Simulation with params {'w_in_amp': np.int64(20), 'gbar': np.int64(100), 'n_input': 3, 'n_output': 3, 'Ne': 400, 'Ni': 100, 'dt': 0.1, 'p_sparsity': 0.1} saved to /home/ali/Code/research/ml_force_research/hpc/bfm_output/test_bfm/w_in_amp_20p0000_gbar_100p0000
Simulation with params {'w_in_amp': np.int64(400), 'gbar': np.int64(100), 'n_input': 3, 'n_output': 3, 'Ne': 400, 'Ni': 100, 'dt': 0.1, 'p_sparsity': 0.1} saved to /home/ali/Code/research/ml_force_research/hpc/bfm_output/test_bfm/w_in_amp_400p0000_gbar_100p0000
Simulation with params {'w_in_amp': np.int64(400), 'gbar': np.int64(5), 'n_input': 3, 'n_output': 3, 'Ne': 400, 'Ni': 100, 'dt': 0.1, 'p_sparsity': 0.1} saved to /home/ali/Code/research/ml_force_research/hpc/bfm_output/test_bfm/w_in_amp_400p0000_gbar_5p0000
Simulation with params {'w_in_amp': np.int64(20), 'gbar': np.int64(5), 'n_input': 3, 'n_output': 3, 'Ne': 400, 'Ni': 100, 'dt': 0.1, 'p_sparsity': 0.1} saved to /home/ali/Code/research/ml_force_research/hpc/bfm_output/te